# <div align="center"> AirNav Indonesia</div>
---

In [1]:
import os
import tarfile
import pandas as pd
import numpy as np

# Konfigurasi file output
filename = "SOETTA.tar.gz"
output_file = "SOETTA_MASTER_FINAL.parquet"

# Hapus file lama kalau ada biar gak tumpuk
if os.path.exists(output_file):
    os.remove(output_file)

print("🚀 Memulai proses 'Save as You Go' (Mode Aman RAM)...")

with tarfile.open(filename, "r:gz") as tar:
    members = [m for m in tar.getmembers() if m.name.endswith('.log')]
    batch_chunks = []
    
    for i, member in enumerate(members):
        f = tar.extractfile(member)
        if f:
            # Baca per baris
            lines = f.read().decode('utf-8').splitlines()
            data_rows = []
            for line in lines:
                if line.startswith('#') or not line.strip(): continue
                parts = line.split('|')
                try:
                    # Ambil data esensial
                    row = {
                        'lon_raw': parts[14], 'lat_raw': parts[15],
                        'icao24': parts[16], 'alt': parts[17],
                        'vertical_rate': parts[34], 'raw_speed': parts[35],
                        'callsign': parts[40], 'timestamp_raw': parts[-1]
                    }
                    data_rows.append(row)
                except IndexError: continue
            
            if data_rows:
                # Bikin DataFrame kecil per file
                temp_df = pd.DataFrame(data_rows).replace('-', np.nan).replace('', np.nan)
                
                # Filter koordinat langsung di sini (Buang sampah secepat mungkin)
                temp_df = temp_df.dropna(subset=['lat_raw', 'lon_raw'])
                
                # Konversi tipe data biar ramping (float32)
                for col in ['lat_raw', 'lon_raw', 'alt', 'vertical_rate', 'raw_speed']:
                    temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce').astype('float32')
                
                batch_chunks.append(temp_df)
        
        # --- KUNCINYA DI SINI: Simpan ke Disk tiap 100 file ---
        if (i + 1) % 100 == 0 or (i + 1) == len(members):
            if batch_chunks:
                df_batch = pd.concat(batch_chunks, ignore_index=True)
                
                # Simpan (append) ke Parquet menggunakan engine 'fastparquet'
                # Note: Jika belum install, run: !pip install fastparquet
                df_batch.to_parquet(
                    output_file, 
                    engine='fastparquet', 
                    append=os.path.exists(output_file)
                )
                
                # KOSONGKAN RAM TOTAL
                batch_chunks = []
                print(f"💾 Progress: {i+1}/{len(members)} file aman di Disk. RAM dikosongkan.")

print(f"✨ BERES! Semua data 30 hari sudah rapi di file: {output_file}")

🚀 Memulai proses 'Save as You Go' (Mode Aman RAM)...


C:\Users\Joel\AppData\Local\Temp\ipykernel_16508\2183451340.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = pd.DataFrame(data_rows).replace('-', np.nan).replace('', np.nan)


💾 Progress: 100/1440 file aman di Disk. RAM dikosongkan.


C:\Users\Joel\AppData\Local\Temp\ipykernel_16508\2183451340.py:42: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  temp_df = pd.DataFrame(data_rows).replace('-', np.nan).replace('', np.nan)


💾 Progress: 200/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 300/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 400/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 500/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 600/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 700/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 800/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 900/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 1000/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 1100/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 1200/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 1300/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 1400/1440 file aman di Disk. RAM dikosongkan.
💾 Progress: 1440/1440 file aman di Disk. RAM dikosongkan.
✨ BERES! Semua data 30 hari sudah rapi di file: SOETTA_MASTER_FINAL.parquet


In [13]:
import pyarrow.parquet as pq

print("🔍 Mengidentifikasi rentang temporal menggunakan metode Stream Metadata...")

# Membuka metadata file Parquet tanpa memuat seluruh data ke memori
parquet_file = pq.ParquetFile(output_file)
unique_dates = set()

print(f"📦 Total Row Groups: {parquet_file.num_row_groups}. Memproses batch...")

# Membaca data secara bertahap (per row group) untuk efisiensi memori
for i in range(parquet_file.num_row_groups):
    # Hanya memuat batch kecil dari kolom timestamp_raw
    batch = parquet_file.read_row_group(i, columns=['timestamp_raw']).to_pandas()
    
    # Ekstraksi tanggal unik dari batch saat ini
    dates = pd.to_datetime(batch['timestamp_raw'], errors='coerce').dt.date.dropna().unique()
    unique_dates.update(dates)
    
    if (i + 1) % 100 == 0:
        print(f"✅ Terproses {i+1} dari {parquet_file.num_row_groups} row groups...")

# Konversi set ke list yang terurut (Sorted Temporal List)
unique_dates = sorted(list(unique_dates))

print(f"\n✨ Identifikasi Selesai.")
print(f"📅 Rentang Operasional: {unique_dates[0]} s/d {unique_dates[-1]}")
print(f"📊 Total Hari Aktif: {len(unique_dates)} hari.")

🔍 Mengidentifikasi rentang temporal menggunakan metode Stream Metadata...
📦 Total Row Groups: 15. Memproses batch...

✨ Identifikasi Selesai.
📅 Rentang Operasional: 2025-03-31 s/d 2025-04-30
📊 Total Hari Aktif: 31 hari.


In [23]:
import pyarrow.parquet as pq

final_ready_file = "SOETTA_MASTER_READY.parquet"

# Prosedur inisialisasi: Membersihkan dataset master lama untuk menjamin integritas data
if os.path.exists(final_ready_file): 
    os.remove(final_ready_file)

print("🛡️ Memulai tahap standarisasi data dengan pendekatan Column-Specific Streaming...")

# Menginisialisasi akses file via PyArrow untuk pembacaan granular
parquet_file = pq.ParquetFile(output_file)

for d in unique_dates:
    print(f"⌛ Memproses data operasional tanggal: {d}...", end=" ")
    
    daily_chunks = []
    
    # Iterasi per Row Group untuk efisiensi alokasi memori
    for i in range(parquet_file.num_row_groups):
        # Tahap 1: Memuat hanya kolom timestamp_raw untuk pengecekan awal (RAM Saver)
        time_col = parquet_file.read_row_group(i, columns=['timestamp_raw']).to_pandas()
        
        # Identifikasi indeks baris yang sesuai dengan kriteria tanggal
        mask = time_col['timestamp_raw'].str.startswith(str(d), na=False)
        
        if mask.any():
            # Tahap 2: Jika data ditemukan, baru muat baris yang relevan untuk kolom lainnya
            # Taktik ini mencegah pemuatan jutaan baris 'sampah' ke RAM
            full_group = parquet_file.read_row_group(i).to_pandas()
            chunk = full_group[mask]
            daily_chunks.append(chunk)
            
            # Dealokasi dataframe besar secepat mungkin
            del full_group
        
        del time_col
            
    if not daily_chunks:
        print("Data tidak ditemukan, melewati...")
        continue
        
    # Konsolidasi batch harian menjadi satu DataFrame operasional
    df_day = pd.concat(daily_chunks, ignore_index=True)
    
    # --- Prosedur Standarisasi Teknis ---
    
    # 1. Perbaikan orientasi koordinat (Swap Field Latitude & Longitude)
    # Note: Berdasarkan profil data JATSC, 107.x adalah Longitude dan -6.x adalah Latitude
    df_day = df_day.rename(columns={'lat_raw': 'lat', 'lon_raw': 'lon'})
    
    # 2. Sinkronisasi format waktu (Time-Series Alignment) untuk analisis kronologis
    df_day['timestamp'] = pd.to_datetime(df_day['timestamp_raw'], errors='coerce')
    
    # 3. Kalkulasi parameter dinamis: Konversi unit NM/s ke metrik KM/H
    df_day['speed_kmh'] = df_day['raw_speed'] * 6667.2
    
    # 4. Prosedur Sorting & Deduplication untuk kebutuhan input model Autoencoder
    df_day = df_day.sort_values(['icao24', 'timestamp']).drop_duplicates()
    
    # 5. Eliminasi atribut redundan untuk optimalisasi penggunaan ruang penyimpanan
    df_day = df_day.drop(columns=['timestamp_raw', 'raw_speed']).dropna(subset=['timestamp'])
    
    # 6. Konsolidasi ke Master Gold Dataset menggunakan engine 'fastparquet' mode append
    df_day.to_parquet(final_ready_file, engine='fastparquet', append=os.path.exists(final_ready_file))
    
    print(f"✅ Selesai. ({len(df_day):,} record tersimpan)")
    
    # Manajemen memori: Dealokasi DataFrame harian sebelum iterasi berikutnya
    del df_day

print(f"\n✨ PIPELINE SELESAI! Master Gold Dataset tersedia di: {final_ready_file}")

🛡️ Memulai tahap standarisasi data dengan pendekatan Column-Specific Streaming...
⌛ Memproses data operasional tanggal: 2025-03-31... ✅ Selesai. (39 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-01... ✅ Selesai. (7,143,427 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-02... ✅ Selesai. (7,158,187 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-03... ✅ Selesai. (7,512,125 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-04... ✅ Selesai. (7,718,571 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-05... ✅ Selesai. (7,630,205 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-06... ✅ Selesai. (8,169,808 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-07... ✅ Selesai. (8,011,387 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-08... ✅ Selesai. (7,806,380 record tersimpan)
⌛ Memproses data operasional tanggal: 2025-04-09... ✅ Selesai. (8,064,739 record tersimpan)
⌛ Mem

In [1]:
import pandas as pd

print("📖 Memuat Master Gold Dataset untuk proses audit final...")
# Memuat dataset yang sudah terstandarisasi
df = pd.read_parquet("SOETTA_MASTER_READY.parquet")

print("🛡️ Menjalankan validasi integritas berdasarkan parameter fisik (Ground Truth)...")

# Threshold: Kecepatan di atas Mach 1 (~1200 km/h) atau Vertical Rate ekstrem (> 3000 ft/min)
# Ini adalah filter dasar untuk mendeteksi potensi data corruption atau awal indikasi spoofing
df['is_speed_anomaly'] = (df['speed_kmh'] > 1200).astype(int)
df['is_altitude_anomaly'] = (df['vertical_rate'].abs() > 3000).astype(int)

# Labeling Anomali Fisik Gabungan
df['is_physical_anomaly'] = ((df['is_speed_anomaly'] == 1) | (df['is_altitude_anomaly'] == 1)).astype(int)

# Simpan kembali dataset dengan label anomali fisik
df.to_parquet("SOETTA_MASTER_READY.parquet", index=False)

print("\n" + "="*50)
print("🏁 DATA ENGINEERING PIPELINE: COMPLETED")
print("="*50)
print(f"📁 Dataset Final: SOETTA_MASTER_READY.parquet")
print(f"📈 Total Record : {len(df):,}")
print(f"🚨 Anomali Fisik: {df['is_physical_anomaly'].sum():,} titik terdeteksi")
print(f"📅 Rentang Waktu: {df['timestamp'].min()} s/d {df['timestamp'].max()}")
print("="*50)

# Menampilkan sampel data dengan indikasi anomali
display(df[df['is_physical_anomaly'] == 1].head())

📖 Memuat Master Gold Dataset untuk proses audit final...
🛡️ Menjalankan validasi integritas berdasarkan parameter fisik (Ground Truth)...

🏁 DATA ENGINEERING PIPELINE: COMPLETED
📁 Dataset Final: SOETTA_MASTER_READY.parquet
📈 Total Record : 235,709,491
🚨 Anomali Fisik: 2,106,845 titik terdeteksi
📅 Rentang Waktu: 2025-03-31 23:59:59.862529 s/d 2025-04-30 23:59:59.146725


,lon,lat,icao24,alt,vertical_rate,callsign,timestamp,speed_kmh,is_speed_anomaly,is_altitude_anomaly,is_physical_anomaly
index,,,,,,,,,,,
6583152,-4.891984,107.111809,0101DB,40175.0,-3006.25,1,2025-04-01 21:57:35.954244,897.698486,0,1,1
6583191,-4.891984,107.111809,0101DB,40175.0,-3006.25,1,2025-04-01 21:57:35.961877,897.698486,0,1,1
6583230,-4.891984,107.111809,0101DB,40175.0,-3006.25,1,2025-04-01 21:57:35.969119,897.698486,0,1,1
6583269,-4.893176,107.112419,0101DB,40150.0,-3006.25,1,2025-04-01 21:57:36.967788,897.698486,0,1,1
6583307,-4.895187,107.113396,0101DB,40075.0,-3075.00,1,2025-04-01 21:57:37.966877,897.698486,0,1,1
